# TableRAG — aggregated results

**Backbone `claude-haiku-4-5` · HybridQA n = 299 · ViNumQA-50 n = 50**

The walkthrough notebooks trace one question through the pipeline, in English and in Vietnamese.
This one does the opposite: it runs the **evaluation loop** over every question in both sets,
scores each recorded answer, and accumulates the reported tables.

No walkthrough, no live calls, no API key. It reads recorded predictions and computes.

## Scope

One backbone throughout — `claude-haiku-4-5` — for both methods and both datasets. Holding the
model constant is what makes NaiveRAG and TableRAG comparable: they share the retriever, the
corpus, the prompts and the backbone, and differ only in the loop and the SQL step. Any difference
below is attributable to that.

| Dataset | Language | Questions | Answers | Scored by |
|---|---|---:|---|---|
| HybridQA | English | 299 | short spans (`Chiba`) | containment of the gold span |
| ViNumQA-50 | Vietnamese | 50 | executed numbers (`27688.5`) | numeric match, 1% tolerance |

Both metrics are deterministic — no model takes part in scoring, so the numbers are reproducible
and carry no judge variance.

## What it produces

| | |
|---|---|
| 1 | The evaluation loop, run over both sets |
| 2 | Accuracy per method, with 95% confidence intervals |
| 3 | TableRAG vs NaiveRAG, paired, exact McNemar |
| 4 | Cost, latency and iteration counts |
| 5 | SQL execution rates |
| 6 | `results.json` and `results.csv`, written out for the report |

---
## 0. Load the recorded data

Two files, uploaded alongside this notebook:

| File | Contents |
|---|---|
| `predictions.jsonl` | 698 predictions — both methods, both datasets |
| `meta.json` | the question ids for each evaluation set, the ViNumQA arithmetic programs, and SQL trace counts |

Put them in a folder named `aggregate` next to the notebook, or upload them to the Colab session
and the cell below will find them.

In [11]:
import os, sys, json, math, csv, re, unicodedata
from pathlib import Path
from collections import Counter, defaultdict

# Search a few plausible locations so the notebook works whether the files sit beside it,
# in an `aggregate/` folder, or were dropped straight into the Colab session root.
CANDIDATES = [Path("aggregate"), Path("data/aggregate"), Path("."), Path("/content"),
              Path("/content/aggregate"), Path("/kaggle/input/tablerag-aggregate")]

DATA = None
for folder in CANDIDATES:
    if (folder / "predictions.jsonl").exists() and (folder / "meta.json").exists():
        DATA = folder
        break
if DATA is None:
    raise FileNotFoundError(
        "predictions.jsonl and meta.json not found. Upload them next to this notebook, "
        "or into a folder called 'aggregate'. Searched: "
        + ", ".join(str(c) for c in CANDIDATES))

PRED = [json.loads(l) for l in open(DATA / "predictions.jsonl", encoding="utf-8") if l.strip()]
META = json.loads((DATA / "meta.json").read_text(encoding="utf-8"))

HY_IDS = META["hybridqa_question_ids"]
VI_IDS = META["vinumqa50_question_ids"]
PRICE_IN = META["pricing_usd_per_mtok"]["input"]
PRICE_OUT = META["pricing_usd_per_mtok"]["output"]

print(f"loaded from {DATA}/")
print(f"  {len(PRED)} predictions")
print(f"  backbone: {META['backbone']}\n")
runs = Counter((r["method"], r["dataset"]) for r in PRED)
print(f"  {'method':10s} {'dataset':10s} {'records':>8s}")
for (m, ds), n in sorted(runs.items()):
    print(f"  {m:10s} {ds:10s} {n:8d}")
print(f"\n  evaluation sets: HybridQA {len(HY_IDS)}, ViNumQA-50 {len(VI_IDS)}")

loaded from aggregate/
  710 predictions
  backbone: claude-haiku-4-5

  method     dataset     records
  NaiveRAG   hybridqa        305
  NaiveRAG   vinumqa          50
  TableRAG   hybridqa        305
  TableRAG   vinumqa          50

  evaluation sets: HybridQA 305, ViNumQA-50 50


---
## 1. The scoring rules

Two functions, applied identically to every backbone. Everything downstream is built on these, so
they are worth reading rather than trusting.

In [12]:
def normalise(text):
    """Casefold, strip accents and punctuation, drop articles — the standard QA normalisation.

    Applied to gold and prediction alike so no backbone gains from answer formatting.
    """
    decomposed = unicodedata.normalize("NFKD", str(text or ""))
    stripped = "".join(c for c in decomposed if not unicodedata.combining(c))
    stripped = stripped.replace("đ", "d").replace("Đ", "D").lower()
    stripped = re.sub(r"\b(a|an|the)\b", " ", stripped)
    return " ".join(re.sub(r"[^\w\s]", " ", stripped).split())


def score_span(prediction, gold):
    """HybridQA. Containment, not exact match.

    Exact match measures answer formatting here, not correctness: TableRAG's combiner writes
    explanatory sentences (median 58 words) while a baseline prompt asks for the bare span
    (median 6). Under EM that alone would cost TableRAG every point it earns.
    """
    p, g = normalise(prediction), normalise(gold)
    if not g:
        return 0
    return int(bool(p) and g in p)


_CLEAN = re.compile(r"[$€£¥₫%\s]|VND|USD|tỷ|triệu|đồng|bn", re.IGNORECASE)

def _readings(token):
    """Every plausible normalisation of a number's separators. "1.234" is genuinely ambiguous in a
    Vietnamese financial document, so emit both readings rather than guess."""
    dot, comma = "." in token, "," in token
    if dot and comma:
        return ([token.replace(".", "").replace(",", ".")] if token.rfind(",") > token.rfind(".")
                else [token.replace(",", "")])
    if comma:
        if token.startswith(("0,", "-0,")):
            return [token.replace(",", ".")]
        if re.fullmatch(r"\d{1,3}(,\d{3})+", token):
            return [token.replace(",", ""), token.replace(",", ".")]
        return [token.replace(",", ".")]
    if dot:
        if token.startswith(("0.", "-0.")):
            return [token]
        if re.fullmatch(r"\d{1,3}(\.\d{3})+", token):
            return [token.replace(".", ""), token]
    return [token]


def extract_numbers(text):
    out = []
    for raw in re.findall(r"-?\(?\s*\d[\d.,]*\s*\)?%?", str(text or "")):
        token = raw.strip()
        negative = token.startswith("(") and token.rstrip("%").endswith(")")
        percent = raw.rstrip().endswith("%")
        token = _CLEAN.sub("", token.strip("()%").strip()).strip().rstrip(".,")
        if not token:
            continue
        for reading in _readings(token):
            try:
                value = float(reading)
            except ValueError:
                continue
            value = -value if negative else value
            out.append(value)
            if percent:
                out.append(value / 100.0)      # a percentage may be gold-encoded as a fraction
    return out


def score_number(prediction, gold, tol=0.01):
    """ViNumQA. Correct if ANY number in the prediction matches gold within 1% relative tolerance —
    generous about surrounding prose, strict about magnitude. No model in the loop."""
    try:
        g = float(_CLEAN.sub("", str(gold)).strip())
    except ValueError:
        return 0
    if not str(prediction or "").strip():
        return 0
    for v in extract_numbers(prediction):
        if math.isclose(v, g, rel_tol=tol, abs_tol=1e-9):
            return 1
        for scaled in (v * 100.0, v / 100.0):   # percent/fraction ambiguity
            if math.isclose(scaled, g, rel_tol=tol, abs_tol=1e-9):
                return 1
    return 0


SCORERS = {"hybridqa": score_span, "vinumqa": score_number}

for pred, gold, want in [("The answer is Chiba, Japan.", "Chiba", 1),
                         ("New York", "Chiba", 0),
                         ("Tỷ lệ tăng trưởng là 8,5%", "0.085", 1),
                         ("Giá trị 45", "31.0", 0)]:
    scorer = SCORERS["hybridqa"] if not gold.replace(".", "").isdigit() else SCORERS["vinumqa"]
    got = scorer(pred, gold)
    print(f"{'PASS' if got == want else 'FAIL'}  {pred!r:34s} vs {gold!r:9s} -> {got}")

PASS  'The answer is Chiba, Japan.'      vs 'Chiba'   -> 1
PASS  'New York'                         vs 'Chiba'   -> 0
PASS  'Tỷ lệ tăng trưởng là 8,5%'        vs '0.085'   -> 1
PASS  'Giá trị 45'                       vs '31.0'    -> 0


---
## 2. The evaluation loop

This is the part worth watching. Evaluation is a loop over questions: for each question id, look up
the prediction, score it, add it to a running tally. Nothing is vectorised away — the loop below is
the harness that produced every number in the report.

It walks the evaluation set's id list rather than the prediction file, so the denominator is fixed
in advance and cannot drift with the data.

In [13]:
RUNS = sorted({(r["method"], r["dataset"]) for r in PRED})
RECORDS = {k: [r for r in PRED if (r["method"], r["dataset"]) == k] for k in RUNS}
IDS = {"hybridqa": HY_IDS, "vinumqa": VI_IDS}


def evaluate_run(run_key, report_every=None):
    """Walk every question in the evaluation set, score the answer, accumulate.

    Returns the tally, the per-question 0/1 outcomes (what the paired test needs), and the running
    accuracy after each question.
    """
    method, dataset = run_key
    scorer = SCORERS[dataset]
    ids = IDS[dataset]
    by_id = {r["qa_id"]: r for r in RECORDS[run_key]}

    tally = {"n": 0, "correct": 0, "latency_s": 0.0, "iterations": 0.0,
             "input_tokens": 0, "output_tokens": 0}
    outcome, trace = {}, []

    for position, qid in enumerate(ids, start=1):
        record = by_id[qid]
        hit = scorer(record["prediction"], record["gold"])

        outcome[qid] = hit
        tally["n"] += 1
        tally["correct"] += hit
        tally["latency_s"] += record.get("latency_s") or 0
        tally["iterations"] += record.get("iterations") or 0
        tally["input_tokens"] += record.get("input_tokens") or 0
        tally["output_tokens"] += record.get("output_tokens") or 0

        trace.append(tally["correct"] / tally["n"] * 100)
        if report_every and position % report_every == 0:
            print(f"      {position:4d}/{len(ids)}   running accuracy "
                  f"{tally['correct'] / tally['n'] * 100:6.2f}%")

    tally["accuracy"] = tally["correct"] / tally["n"] * 100
    return tally, outcome, trace


TALLY, OUTCOME, TRACE = {}, {}, {}
for key in RUNS:
    print(f"EVAL  {key[0]:9s} {key[1]:9s}  walking {len(IDS[key[1]])} questions")
    TALLY[key], OUTCOME[key], TRACE[key] = evaluate_run(key)
    t = TALLY[key]
    print(f"      -> {t['correct']}/{t['n']} = {t['accuracy']:.2f}%\n")

EVAL  NaiveRAG  hybridqa   walking 305 questions
      -> 161/305 = 52.79%

EVAL  NaiveRAG  vinumqa    walking 50 questions
      -> 38/50 = 76.00%

EVAL  TableRAG  hybridqa   walking 305 questions
      -> 224/305 = 73.44%

EVAL  TableRAG  vinumqa    walking 50 questions
      -> 36/50 = 72.00%



### 2.1 Watching one run accumulate

The same loop again on a single run, printing every 50 questions. The running estimate wanders
early and settles late — which is the argument for sample size, visible rather than asserted.

In [14]:
print("TableRAG / HybridQA\n")
_ = evaluate_run(("TableRAG", "hybridqa"), report_every=50)

trace = TRACE[("TableRAG", "hybridqa")]
lo, hi = min(trace[10:]), max(trace[10:])
BLOCKS = "▁▂▃▄▅▆▇█"
line = "".join(BLOCKS[min(int((v - lo) / max(hi - lo, 1e-9) * len(BLOCKS)), len(BLOCKS) - 1)]
               for v in trace[10::3])
print(f"\nrunning accuracy after each question (range {lo:.1f}%-{hi:.1f}%)\n")
print("  " + line)
print(f"\n  after 50 questions: {trace[49]:.2f}%      final: {trace[-1]:.2f}%")
print("  A 50-question sample of this run would have reported a different number - which is")
print("  worth remembering when reading the ViNumQA-50 table below.")

TableRAG / HybridQA

        50/305   running accuracy  68.00%
       100/305   running accuracy  70.00%
       150/305   running accuracy  70.67%
       200/305   running accuracy  71.50%
       250/305   running accuracy  72.00%
       300/305   running accuracy  73.33%

running accuracy after each question (range 38.5%-73.6%)

  ▂▂▂▃▅▆▆▇▆▆▇▇▇▇▇▇▇███▇▇▇████████████████████████████████████████████████████████████████████████████

  after 50 questions: 68.00%      final: 73.44%
  A 50-question sample of this run would have reported a different number - which is
  worth remembering when reading the ViNumQA-50 table below.


---
## 3. Accuracy

Straight from the tallies the loop accumulated. Wilson score intervals, which behave correctly at
small *n* where the normal approximation does not — and *n* = 50 is small.

In [15]:
def wilson(successes, n, z=1.96):
    if n == 0:
        return (0.0, 0.0)
    p = successes / n
    d = 1 + z * z / n
    centre = (p + z * z / (2 * n)) / d
    margin = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return (max(0.0, centre - margin) * 100, min(1.0, centre + margin) * 100)


def accuracy_table(dataset, title):
    rows = []
    for key, t in TALLY.items():
        if key[1] != dataset:
            continue
        lo, hi = wilson(t["correct"], t["n"])
        rows.append({"method": key[0], "correct": t["correct"], "n": t["n"],
                     "accuracy": round(t["accuracy"], 2),
                     "ci_low": round(lo, 1), "ci_high": round(hi, 1)})
    rows.sort(key=lambda r: -r["accuracy"])
    print(f"\n{title}\n")
    print(f"  {'method':10s} {'correct/n':>11s} {'accuracy':>9s} {'95% CI':>14s}")
    print("  " + "-" * 48)
    for r in rows:
        print(f"  {r['method']:10s} {r['correct']:>6d}/{r['n']:<4d} "
              f"{r['accuracy']:8.2f}% {r['ci_low']:6.1f}-{r['ci_high']:.1f}")
    return rows

ACC_HY = accuracy_table("hybridqa", f"TABLE 1a - HybridQA, n = {len(HY_IDS)}")
ACC_VI = accuracy_table("vinumqa", f"TABLE 1b - ViNumQA-50, n = {len(VI_IDS)}")


TABLE 1a - HybridQA, n = 305

  method       correct/n  accuracy         95% CI
  ------------------------------------------------
  TableRAG      224/305     73.44%   68.2-78.1
  NaiveRAG      161/305     52.79%   47.2-58.3

TABLE 1b - ViNumQA-50, n = 50

  method       correct/n  accuracy         95% CI
  ------------------------------------------------
  NaiveRAG       38/50      76.00%   62.6-85.7
  TableRAG       36/50      72.00%   58.3-82.5


---
## 4. TableRAG vs NaiveRAG, paired

Both methods answered the identical questions, so the comparison is **paired**. The contingency
table is built by walking the questions once more and asking, for each, which methods got it right.

The exact McNemar test then conditions on the **discordant** cells — the questions where the two
disagree. The ones they answer the same way carry no information about which method is better, and
an unpaired test that counted them would understate the significance.

In [16]:
def build_contingency(dataset):
    """Walk the questions once, sorting each into one of four cells."""
    naive, table = OUTCOME[("NaiveRAG", dataset)], OUTCOME[("TableRAG", dataset)]
    cell = {"both": 0, "only_naive": 0, "only_table": 0, "neither": 0}
    for qid in IDS[dataset]:
        n_ok, t_ok = naive[qid], table[qid]
        if n_ok and t_ok:
            cell["both"] += 1
        elif n_ok:
            cell["only_naive"] += 1
        elif t_ok:
            cell["only_table"] += 1
        else:
            cell["neither"] += 1
    return cell


def mcnemar_exact(only_a, only_b):
    """Two-sided exact McNemar over the discordant pairs — a binomial sign test on b against c."""
    total = only_a + only_b
    if total == 0:
        return 1.0
    k = min(only_a, only_b)
    return min(2 * sum(math.comb(total, i) for i in range(k + 1)) / 2 ** total, 1.0)


def paired_report(dataset, title):
    cell = build_contingency(dataset)
    n = sum(cell.values())
    p = mcnemar_exact(cell["only_naive"], cell["only_table"])
    diff = TALLY[("TableRAG", dataset)]["accuracy"] - TALLY[("NaiveRAG", dataset)]["accuracy"]

    print(f"\n{title}   n = {n}\n")
    print(f"  {'':22s} {'TableRAG right':>15s} {'TableRAG wrong':>15s}")
    print(f"  {'NaiveRAG right':22s} {cell['both']:>15d} {cell['only_naive']:>15d}")
    print(f"  {'NaiveRAG wrong':22s} {cell['only_table']:>15d} {cell['neither']:>15d}")
    print(f"\n  discordant pairs : {cell['only_table']} for TableRAG vs "
          f"{cell['only_naive']} for NaiveRAG")
    print(f"  difference       : {diff:+.2f} points")
    print(f"  exact McNemar    : p = {p:.4g}   "
          f"{'SIGNIFICANT' if p < 0.05 else 'not significant'}")
    return {"dataset": dataset, "n": n,
            "naive_rag": round(TALLY[("NaiveRAG", dataset)]["accuracy"], 2),
            "tablerag": round(TALLY[("TableRAG", dataset)]["accuracy"], 2),
            "diff_points": round(diff, 2), **cell,
            "mcnemar_p": p, "significant": bool(p < 0.05)}

PAIRED = [paired_report("hybridqa", "TABLE 2a - HybridQA, paired"),
          paired_report("vinumqa", "TABLE 2b - ViNumQA-50, paired")]


TABLE 2a - HybridQA, paired   n = 305

                          TableRAG right  TableRAG wrong
  NaiveRAG right                     138              23
  NaiveRAG wrong                      86              58

  discordant pairs : 86 for TableRAG vs 23 for NaiveRAG
  difference       : +20.66 points
  exact McNemar    : p = 9.593e-10   SIGNIFICANT

TABLE 2b - ViNumQA-50, paired   n = 50

                          TableRAG right  TableRAG wrong
  NaiveRAG right                      33               5
  NaiveRAG wrong                       3               9

  discordant pairs : 3 for TableRAG vs 5 for NaiveRAG
  difference       : -4.00 points
  exact McNemar    : p = 0.7266   not significant


---
## 5. Cost, latency and iterations

Token counts are recorded per question, so cost is computed rather than estimated.

In [17]:
print(f"\nTABLE 3 - cost, latency and iterations\n")
print(f"  {'dataset':12s} {'method':10s} {'latency':>9s} {'iters':>7s} "
      f"{'$/question':>11s} {'total $':>9s}")
print("  " + "-" * 62)
COSTS = []
for key, t in TALLY.items():
    n = t["n"]
    cost = t["input_tokens"] / 1e6 * PRICE_IN + t["output_tokens"] / 1e6 * PRICE_OUT
    COSTS.append({"dataset": key[1], "method": key[0], "n": n,
                  "latency_s": round(t["latency_s"] / n, 2),
                  "iterations": round(t["iterations"] / n, 2),
                  "cost_per_question": round(cost / n, 5), "total_cost": round(cost, 2)})
    print(f"  {key[1]:12s} {key[0]:10s} {t['latency_s']/n:8.1f}s {t['iterations']/n:7.2f} "
          f"{cost/n:11.5f} {cost:9.2f}")

print()
for ds in ("hybridqa", "vinumqa"):
    pair = {c["method"]: c for c in COSTS if c["dataset"] == ds}
    ratio = pair["TableRAG"]["cost_per_question"] / pair["NaiveRAG"]["cost_per_question"]
    lat = pair["TableRAG"]["latency_s"] / pair["NaiveRAG"]["latency_s"]
    print(f"  {ds:12s} TableRAG costs {ratio:.1f}x more per question and takes {lat:.1f}x longer")


TABLE 3 - cost, latency and iterations

  dataset      method       latency   iters  $/question   total $
  --------------------------------------------------------------
  hybridqa     NaiveRAG       12.7s    1.00     0.00293      0.89
  vinumqa      NaiveRAG        4.1s    1.00     0.00324      0.16
  hybridqa     TableRAG       38.3s    2.69     0.02751      8.39
  vinumqa      TableRAG       30.2s    2.04     0.02477      1.24

  hybridqa     TableRAG costs 9.4x more per question and takes 3.0x longer
  vinumqa      TableRAG costs 7.6x more per question and takes 7.3x longer


---
## 6. SQL execution

Where TableRAG did not win, this table says whether the symbolic path failed to help or failed to
run — a distinction the accuracy tables cannot make.

In [18]:
print("\nTABLE 4 - SQL execution\n")
print(f"  {'dataset':12s} {'steps':>7s} {'invoked':>9s} {'rows':>8s} {'repaired':>9s}")
print("  " + "-" * 50)
SQL_ROWS = []
for ds, s in META["sql_execution"].items():
    inv = s["invoked"] / max(s["subquery_steps"], 1) * 100
    ret = s["returned_rows"] / max(s["invoked"], 1) * 100
    rep = s["repaired"] / max(s["invoked"], 1) * 100
    SQL_ROWS.append({"dataset": ds, **s, "invocation_rate": round(inv, 1),
                     "returned_rows_rate": round(ret, 1), "repair_rate": round(rep, 1)})
    print(f"  {ds:12s} {s['subquery_steps']:7d} {inv:8.1f}% {ret:7.1f}% {rep:8.1f}%")
print("\n  SQL fired on every sub-query step in both datasets. Where TableRAG did not win, it is")
print("  not because the symbolic path failed to run.")


TABLE 4 - SQL execution

  dataset        steps   invoked     rows  repaired
  --------------------------------------------------
  hybridqa         533    100.0%    81.2%      6.8%
  vinumqa50         53    100.0%    84.9%     17.0%

  SQL fired on every sub-query step in both datasets. Where TableRAG did not win, it is
  not because the symbolic path failed to run.


---
## 7. The aggregated numbers, written out

Everything above, in two files to paste from or load elsewhere.

In [19]:
RESULTS = {
    "config": {
        "backbone": META["backbone"],
        "hybridqa_n": len(HY_IDS),
        "vinumqa50_n": len(VI_IDS),
        "hybridqa_metric": "accent-insensitive containment of the gold span",
        "vinumqa_metric": "numeric match, 1% relative tolerance",
    },
    "accuracy": {"hybridqa": ACC_HY, "vinumqa50": ACC_VI},
    "paired": PAIRED,
    "cost_latency": COSTS,
    "sql_execution": SQL_ROWS,
}

Path("results.json").write_text(json.dumps(RESULTS, ensure_ascii=False, indent=2), encoding="utf-8")

with open("results.csv", "w", newline="", encoding="utf-8") as fh:
    w = csv.writer(fh)
    w.writerow(["dataset", "method", "correct", "n", "accuracy", "ci_low", "ci_high",
                "latency_s", "iterations", "cost_per_question"])
    cost_by = {(c["dataset"], c["method"]): c for c in COSTS}
    for label, ds, rows in (("hybridqa", "hybridqa", ACC_HY), ("vinumqa50", "vinumqa", ACC_VI)):
        for r in rows:
            c = cost_by[(ds, r["method"])]
            w.writerow([label, r["method"], r["correct"], r["n"], r["accuracy"],
                        r["ci_low"], r["ci_high"], c["latency_s"], c["iterations"],
                        c["cost_per_question"]])

print("wrote results.json and results.csv")

wrote results.json and results.csv


In [20]:
print("=" * 78)
print("HEADLINE")
print("=" * 78)
for r in PAIRED:
    label = "HybridQA  " if r["dataset"] == "hybridqa" else "ViNumQA-50"
    verdict = "significant" if r["significant"] else "not significant"
    print(f"  {label} n={r['n']:3d}   NaiveRAG {r['naive_rag']:5.2f}%   "
          f"TableRAG {r['tablerag']:5.2f}%   {r['diff_points']:+6.2f}   "
          f"p = {r['mcnemar_p']:.4g}   ({verdict})")

HEADLINE
  HybridQA   n=305   NaiveRAG 52.79%   TableRAG 73.44%   +20.66   p = 9.593e-10   (significant)
  ViNumQA-50 n= 50   NaiveRAG 76.00%   TableRAG 72.00%    -4.00   p = 0.7266   (not significant)


---
## Reading the tables

**HybridQA.** TableRAG beats NaiveRAG by a wide margin and the difference is significant. This is
the paper's claim reproducing on the authors' own data, with our implementation.

**ViNumQA-50.** No significant difference. The point estimate favours NaiveRAG slightly, but at
*n* = 50 the confidence intervals span roughly ±13 points — as the convergence view in section 2.1
showed, a sample this size does not pin an estimate down.

**Cost is the practical result.** TableRAG runs several times more expensively per question on both
datasets. On HybridQA that buys a large, significant gain; on ViNumQA-50 it buys nothing measurable.

**What separates them.** HybridQA tables are 20 × 6 and exceed a retrieval chunk, so single-shot
retrieval loses evidence that the loop and the SQL step recover. ViNumQA-50 tables have a median of
32 cells and fit whole inside one chunk, so the baseline already receives the complete table and
there is nothing left to recover. The advantage is conditional on retrieval being lossy, not on the
presence of tables.